In [ ]:
import os

from decimal import Decimal
import numpy as np
from numpy import genfromtxt
from numpy import savetxt
import glob
import re
import importlib
import gc
import matplotlib.pyplot as plt
import math
import memory_profiler
from time import time

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import scale
from sklearn.linear_model import LinearRegression
import pandas as pd

import keras
from keras.models import load_model
from keras.models import Sequential, load_model
from keras.layers.core import Dense, Flatten
from keras.layers import Input, Concatenate
from keras.callbacks import ModelCheckpoint, EarlyStopping

from platform import python_version

import tensorflow as tf
from tensorflow.keras.models import clone_model

import optuna
import random
import shap

rmse = tf.keras.metrics.RootMeanSquaredError()

In [ ]:
print('Loading original model and activation function:')
leakyrelu=lambda x: tf.keras.activations.relu(x, alpha=0.2)
Original=load_model("./originalModel.h5",custom_objects={'<lambda>': leakyrelu})

In [ ]:
print('Loading orignal dataset:')
ds=[]
orden=[]
total_samples = 0
for filename in glob.glob("./dataSet/NNTrainingTesting-3p-3c-v7_*.csv"):
    temp=filename[43:-1].split("_")
    DP=temp[:1]
    DP=''.join(DP)
    DP=int(DP)
    orden.append(DP)
    if DP >= 100:
        params = np.array(filename[49+2:-4].split("_"), dtype=np.float32)
    elif DP >= 10:
        params = np.array(filename[49+1:-4].split("_"), dtype=np.float32)
    else:
        params = np.array(filename[49:-4].split("_"), dtype=np.float32)  
    data = genfromtxt(filename, delimiter='\n')
    row = np.concatenate((data, params), axis=0)
    ds.append(row)
    total_samples+=1
    
print("Total samples: ", total_samples)

ds = np.array(ds, dtype=np.float32)
orden=np.array(orden, dtype=int)

print(ds.shape)

In [ ]:
labels=np.array(ds[:,600:])/5000
data=np.array(ds[:,:600])
inputs=data.shape[1]

scaler=StandardScaler()
scaler.fit(data)

train_final_input = data[:400,:]
test_final_input = data[400:,:]
train_final_label=labels[:400,:]
test_final_label=labels[400:,:]

train_final_input=scaler.transform(train_final_input)
test_final_input=scaler.transform(test_final_input)

trainfi_Vx=np.array(train_final_input[:,0:inputs-2:3])
trainfi_Vy=np.array(train_final_input[:,1:inputs-1:3])
trainfi_Vz=np.array(train_final_input[:,2:inputs:3])

testfi_Vx=np.array(test_final_input[:,0:inputs-2:3])
testfi_Vy=np.array(test_final_input[:,1:inputs-1:3])
testfi_Vz=np.array(test_final_input[:,2:inputs:3])

trainfi_Vx_pd=pd.DataFrame(trainfi_Vx)
trainfi_Vy_pd=pd.DataFrame(trainfi_Vy)
trainfi_Vz_pd=pd.DataFrame(trainfi_Vz)
testfi_Vx_pd=pd.DataFrame(testfi_Vx)
testfi_Vy_pd=pd.DataFrame(testfi_Vy)
testfi_Vz_pd=pd.DataFrame(testfi_Vz)


trainfl_pd=pd.DataFrame(train_final_label)
testfl_pd=pd.DataFrame(test_final_label)

In [ ]:
%load_ext memory_profiler

In [ ]:
%%memit
start=time()
explainer = shap.GradientExplainer(Original, [trainfi_Vx,trainfi_Vy,trainfi_Vz])
shap_values = explainer.shap_values([testfi_Vx,testfi_Vy,testfi_Vz])
shap_values=np.array(shap_values)
shap_values=shap_values.reshape(3,100,-1)
#np.savez('./shapValues.npz',shap_values)
data=[i**2 for i in range(10_000_000)]
print(f'execution time: {time()-start}')

In [ ]:
shap_values=np.load('./shapValues.npz')['arr_0']
shap_values.shape

In [ ]:
weight=np.mean(abs(shap_values),axis=(0,1))

In [ ]:
plt.hist(weight,20)
plt.show()

In [ ]:
#Select inputs with mean higher than a given threslhold
features=np.zeros((600,2))
features[:,0]=weight
features[:,1]=np.linspace(0,599,600,dtype=int)
sorted=features[features[:, 0].argsort()]
#Number of selected inputs
nSelected=5
#180 per a 100 inputs
vips=sorted[-nSelected:,1]
vips=vips.astype(int)

In [ ]:
#Locate the input to represent it

componente=[]
fila=[]
punto=[]

for i in range(len(vips)):
    resto=np.remainder(vips[i],3)
    componente.append(resto+1)
    for j in range(10):
        if (vips[i]<(600-60*j))&(vips[i]>=(600-60*(j+1))):
            fila.append(j)
    for z in range(20):
        if (vips[i]>(60*(9-fila[i])+3*z))&(vips[i]<=(60*(9-fila[i])+3*(z+1))):
            punto.append(z)
  

In [ ]:
#Print inputs' locations

deltax=0.105
deltaz=0.5555558

posicion=np.zeros((len(vips),2))
for i in range(len(vips)):
    posicion[i,0]=-punto[i]*deltax
    posicion[i,1]=-2.5+fila[i]*deltaz
    
    
line1=np.array([[-5,7],[0,0]])
line2=np.array([[0,0],[0.5,-4]])
plt.scatter(posicion[:,1],posicion[:,0],color='r')
plt.plot(line1[0],line1[1],'k--')
plt.plot(line2[0],line2[1],'k--')
plt.ylim(0.5,-4)
plt.xlim(7,-5)
plt.xlabel('z [m]',fontsize=15)
plt.ylabel('r [m]',fontsize=15)
plt.savefig('./reviewFigures/location_M8.pdf')

In [ ]:
#Complete the sampling point with the remaining velocity components of each selected input

datos=[]
completos=np.ones(len(vips)*3)*-1
for i in range(len(vips)):
    resto=np.remainder(vips[i],3)
    if (vips[i]==completos).any():
        print('Sampling point repeated')
    else:
        if resto==0:
            col=[ds[:,vips[i]],ds[:,vips[i]+1],ds[:,vips[i]+2]]
            col=np.array(col,dtype=float)
            col2=col.transpose()
            completos[3*i]=vips[i]
            completos[3*i+1]=vips[i]+1
            completos[3*i+2]=vips[i]+2
        elif resto==1:
            col=[ds[:,vips[i]-1],ds[:,vips[i]],ds[:,vips[i]+1]]
            col=np.array(col,dtype=float)
            col2=col.transpose()
            completos[3*i]=vips[i]-1
            completos[3*i+1]=vips[i]
            completos[3*i+2]=vips[i]+1
        elif resto==2:
            col=[ds[:,vips[i]-2],ds[:,vips[i]-1],ds[:,vips[i]]]
            col=np.array(col,dtype=float)
            col2=col.transpose()
            completos[3*i]=vips[i]-2
            completos[3*i+1]=vips[i]-1
            completos[3*i+2]=vips[i]
        if i==0:
            datos=col2
        else:
            datos=np.append(datos,col2,axis=1)

            datos=np.array(datos,dtype=float)
            
print(datos.shape)
inputs=datos.shape[1]
print(inputs/3)

labels=np.array(ds[:,600:])/5000

scaler=StandardScaler()
scaler.fit(datos)

## Hyperparameter optimization

In [ ]:
#Reduce dataset leaving only the selected sampling points
train_final_input = datos[:400,:]
test_final_input = datos[400:,:]
train_final_label=labels[:400,:]
test_final_label=labels[400:,:]

train_final_input=scaler.transform(train_final_input)
test_final_input=scaler.transform(test_final_input)

trainfi_Vx=np.array(train_final_input[:,0:inputs-2:3])
trainfi_Vy=np.array(train_final_input[:,1:inputs-1:3])
trainfi_Vz=np.array(train_final_input[:,2:inputs:3])

testfi_Vx=np.array(test_final_input[:,0:inputs-2:3])
testfi_Vy=np.array(test_final_input[:,1:inputs-1:3])
testfi_Vz=np.array(test_final_input[:,2:inputs:3])

trainfi_Vx_pd=pd.DataFrame(trainfi_Vx)
trainfi_Vy_pd=pd.DataFrame(trainfi_Vy)
trainfi_Vz_pd=pd.DataFrame(trainfi_Vz)
testfi_Vx_pd=pd.DataFrame(testfi_Vx)
testfi_Vy_pd=pd.DataFrame(testfi_Vy)
testfi_Vz_pd=pd.DataFrame(testfi_Vz)


trainfl_pd=pd.DataFrame(train_final_label)
testfl_pd=pd.DataFrame(test_final_label)

In [ ]:
def set_seed(seed):
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()  # Optional: For deterministic ops
    random.seed(seed)
    np.random.seed(seed)

def reinitialize_weights(model, seed):
    set_seed(seed)

    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):  # Recursively handle nested models
            reinitialize_weights(layer, seed)
            continue

        for attr in ['kernel', 'bias', 'recurrent_kernel']:
            if hasattr(layer, attr):
                var = getattr(layer, attr)
                initializer = getattr(layer, f"{attr}_initializer", None)
                if initializer is not None:
                    config = initializer.get_config()

                    # Only add seed if the initializer supports it
                    if 'seed' in config:
                        config['seed'] = seed

                    # Recreate the initializer safely
                    new_initializer = initializer.__class__.from_config(config)

                    # Assign new initialized value
                    var.assign(new_initializer(var.shape, var.dtype))

In [ ]:
def objective(trial):

    learning_rate = trial.suggest_loguniform('learning_rate',1e-6, 1e-3)
    batch_size = trial.suggest_int('batch_size', 1, 32)
    val_split = trial.suggest_uniform('val_split',0.1,0.5)

    #Create MLP with initial layer adapted to new number of inputs
    I1=Input(shape=(trainfi_Vx.shape[1],))
    I2=Input(shape=(trainfi_Vy.shape[1],))
    I3=Input(shape=(trainfi_Vz.shape[1],))

    h1_p1=Dense(trainfi_Vx.shape[1],activation=leakyrelu)(I1)
    h1_p2=Dense(trainfi_Vy.shape[1],activation=leakyrelu)(I2)
    h1_p3=Dense(trainfi_Vz.shape[1],activation=leakyrelu)(I3)
    h1=Concatenate()([h1_p1,h1_p2,h1_p3])
    h2=Dense(500,activation=leakyrelu)(h1)
    hdp1=keras.layers.Dropout(0.5)(h2)
    h3=Dense(420,activation=leakyrelu)(hdp1)
    hdp2=keras.layers.Dropout(0.35)(h3)
    h4=Dense(360,activation=leakyrelu)(hdp2)
    hdp3=keras.layers.Dropout(0.25)(h4)
    h5=Dense(300,activation=leakyrelu)(hdp3)
    hdp4=keras.layers.Dropout(0.15)(h5)
    h6=Dense(150,activation=leakyrelu)(hdp4)
    h7=Dense(75,activation=leakyrelu)(h6)
    h8=Dense(32,activation=leakyrelu)(h7)
    Out=Dense(3)(h8)

    model=keras.models.Model(inputs=[I1,I2,I3],outputs=Out)
    model.compile(loss='mae', optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

    set_seed(73)
    reinitialize_weights(model, 73)

    earlystop = EarlyStopping(monitor='val_loss', patience=100, min_delta=0.0005, verbose=1, restore_best_weights = False)
    callbacks_list = [earlystop]

    history = model.fit([trainfi_Vx_pd,trainfi_Vy_pd,trainfi_Vz_pd], trainfl_pd,
    epochs=10000, batch_size=batch_size, verbose=0, validation_split = val_split,callbacks=callbacks_list)

    loss=model.evaluate([testfi_Vx_pd,testfi_Vy_pd,testfi_Vz_pd], testfl_pd)

    del model

    return loss

In [ ]:
leakyrelu=lambda x: tf.keras.activations.relu(x, alpha=0.2)
start=time()
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

mins=(time()-start)/60
print(mins)

best_params = study.best_params
best_loss = study.best_value
print("Best Parameters:", best_params)
print("Best Loss:", best_loss)
saveParams=np.zeros(3)
saveParams[0]=best_params['learning_rate']
saveParams[1]=best_params['batch_size']
saveParams[2]=best_params['val_split']
np.savez('./bestHyperparams/method8.npz',saveParams)

## Training of the 5 independent models

In [ ]:
best_params=np.load('./bestHyperparams/method8.npz')['arr_0']
print(best_params)

In [ ]:
@tf.keras.utils.register_keras_serializable()
def leakyrelu(x):
    return tf.keras.activations.relu(x, alpha=0.2)

In [ ]:
def random_split(inputs,labels,seed):
    np.random.seed(seed)

    indices = np.arange(inputs.shape[0])
    np.random.shuffle(indices)

    train_size = 400
    train_indices = indices[:train_size]
    test_indices = indices[train_size:]

    train_final_input = datos[train_indices]
    test_final_input = datos[test_indices]
    train_final_label=labels[train_indices]
    test_final_label=labels[test_indices]

    return train_final_input,test_final_input,train_final_label,test_final_label

In [ ]:
#Train model with best parameters
loses=np.ones(5)*1000
m=np.zeros((5,3))
R2=np.zeros((5,3))
seedWeights=73
seedsCases=[13,37,258,456,1379]
labelSeed=np.empty((1,3))
predSeed=np.empty((1,3))

for i in range(5):
    #Select different cases for train and test according to seed
    train_final_input,test_final_input,train_final_label,test_final_label = random_split(datos,labels,seedsCases[i])

    train_final_input=scaler.transform(train_final_input)
    test_final_input=scaler.transform(test_final_input)

    trainfi_Vx=np.array(train_final_input[:,0:inputs-2:3])
    trainfi_Vy=np.array(train_final_input[:,1:inputs-1:3])
    trainfi_Vz=np.array(train_final_input[:,2:inputs:3])

    testfi_Vx=np.array(test_final_input[:,0:inputs-2:3])
    testfi_Vy=np.array(test_final_input[:,1:inputs-1:3])
    testfi_Vz=np.array(test_final_input[:,2:inputs:3])

    trainfi_Vx_pd=pd.DataFrame(trainfi_Vx)
    trainfi_Vy_pd=pd.DataFrame(trainfi_Vy)
    trainfi_Vz_pd=pd.DataFrame(trainfi_Vz)
    testfi_Vx_pd=pd.DataFrame(testfi_Vx)
    testfi_Vy_pd=pd.DataFrame(testfi_Vy)
    testfi_Vz_pd=pd.DataFrame(testfi_Vz)

    trainfl_pd=pd.DataFrame(train_final_label)
    testfl_pd=pd.DataFrame(test_final_label)

    #Create MLP with initial layer adapted to new number of inputs
    I1=Input(shape=(trainfi_Vx.shape[1],))
    I2=Input(shape=(trainfi_Vy.shape[1],))
    I3=Input(shape=(trainfi_Vz.shape[1],))

    h1_p1=Dense(trainfi_Vx.shape[1],activation=leakyrelu)(I1)
    h1_p2=Dense(trainfi_Vy.shape[1],activation=leakyrelu)(I2)
    h1_p3=Dense(trainfi_Vz.shape[1],activation=leakyrelu)(I3)
    h1=Concatenate()([h1_p1,h1_p2,h1_p3])
    h2=Dense(500,activation=leakyrelu)(h1)
    hdp1=keras.layers.Dropout(0.5)(h2)
    h3=Dense(420,activation=leakyrelu)(hdp1)
    hdp2=keras.layers.Dropout(0.35)(h3)
    h4=Dense(360,activation=leakyrelu)(hdp2)
    hdp3=keras.layers.Dropout(0.25)(h4)
    h5=Dense(300,activation=leakyrelu)(hdp3)
    hdp4=keras.layers.Dropout(0.15)(h5)
    h6=Dense(150,activation=leakyrelu)(hdp4)
    h7=Dense(75,activation=leakyrelu)(h6)
    h8=Dense(32,activation=leakyrelu)(h7)
    Out=Dense(3)(h8)

    model=keras.models.Model(inputs=[I1,I2,I3],outputs=Out)
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=best_params[0]), metrics=['mae'])

    set_seed(seedWeights)
    reinitialize_weights(model, seedWeights)

    earlystop = EarlyStopping(monitor='val_loss', patience=100, min_delta=0.0005, verbose=1, restore_best_weights = False)
    callbacks_list = [earlystop]

    history = model.fit([trainfi_Vx_pd,trainfi_Vy_pd,trainfi_Vz_pd], trainfl_pd,epochs=10000,
    batch_size=int(best_params[1]),verbose=0,validation_split = best_params[2],callbacks=callbacks_list)

    loss=model.evaluate([testfi_Vx_pd,testfi_Vy_pd,testfi_Vz_pd], testfl_pd)
    predicciones=model.predict([testfi_Vx_pd,testfi_Vy_pd,testfi_Vz_pd])
    for j in range(3):
        regression=LinearRegression().fit(test_final_label[:,j].reshape(-1,1),predicciones[:,j].reshape(-1,1))
        R2[i,j]=regression.score(test_final_label[:,j].reshape(-1,1),predicciones[:,j].reshape(-1,1))
        m[i,j]=regression.coef_
        
    if (loss[1]<np.min(loses)):
        best_model = clone_model(model)
        best_model.build(model.input_shape)
        best_model.set_weights(model.get_weights())
        loses[i]=loss[1]
    else:
        del model
        loses[i]=loss[1]

    predSeed=np.append(predSeed,predicciones,axis=0)
    labelSeed=np.append(labelSeed,test_final_label,axis=0)

In [ ]:
def plot_loss(history,maximo):
  plt.figure(figsize=(16,8))
  plt.plot(history.history['loss'], label='loss')
  plt.plot(history.history['val_loss'], label='val_loss')
  plt.ylim([0, maximo])
  plt.xlabel('Epoch')
  plt.ylabel('Error')
  plt.legend()
  plt.grid(True)

In [ ]:
#Print training

hist = pd.DataFrame(history.history)
hist['epoch'] = history.epoch
plot_loss(history,0.02)

In [ ]:
#Evaluate model
best_model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=best_params[0]), metrics=['mae'])
results= best_model.evaluate([testfi_Vx_pd,testfi_Vy_pd,testfi_Vz_pd], testfl_pd, verbose=1)

In [ ]:
print(loses)
print(np.mean(loses),np.std(loses))
print(np.mean(m),np.std(m))
print(np.mean(R2),np.std(R2))

In [ ]:
#Create maeANDstd file for this method
newData=np.zeros((1,6))
newData[0,0]=np.mean(loses)
newData[0,1]=np.std(loses)
newData[0,2]=np.mean(m)
newData[0,3]=np.std(m)
newData[0,4]=np.mean(R2)
newData[0,5]=np.std(R2)
np.savez('./statistics/method8.npz',newData)

In [ ]:
#Add new mae and std to the existing file
newData=np.zeros((1,6))
newData[0,0]=np.mean(loses)
newData[0,1]=np.std(loses)
newData[0,2]=np.mean(m)
newData[0,3]=np.std(m)
newData[0,4]=np.mean(R2)
newData[0,5]=np.std(R2)
oldData=np.load('./statistics/method8.npz')['arr_0']
oldData=np.append(oldData,newData,axis=0)
np.savez('./statistics/method8.npz',oldData)

## Save and plot predictions

In [ ]:
together=np.concatenate((predSeed[1:],labelSeed[1:]),axis=1)
np.savez('./preds_5seeds/m8_2inputs.npz',together)

In [ ]:
xlinea=np.linspace(0,5000,100)
origen=np.ones((1,100))

fig, axs = plt.subplots(1, 3,figsize=(15,4))
inputs=['5 inputs', '3 inputs', '1 input']

for i in range(3):
    allPreds=np.load(f'./preds_5seeds/m8_{5-i*2}inputs.npz')['arr_0'][:,:3]*5000
    allLabels=np.load(f'./preds_5seeds/m8_{5-i*2}inputs.npz')['arr_0'][:,3:]*5000
    regZZ=LinearRegression().fit(allLabels[:,2].reshape(-1,1),allPreds[:,2].reshape(-1,1))
    origen=origen*regZZ.intercept_
    ylinea=origen+xlinea*regZZ.coef_

    axs[i].scatter(allLabels[:,2].reshape(-1,1),allPreds[:,2].reshape(-1,1),20)
    axs[i].plot(xlinea,xlinea,'g--')
    axs[i].plot(xlinea,ylinea.reshape(-1,1),'r.',markersize=1)
    origen=np.ones((1,100))
    axs[i].set_xlabel(r' $M_z$ (kg $m^-2$ $s^-2$)',fontsize=15)
    axs[i].set_ylabel(r' $M_z^\prime$ (kg $m^-2$ $s^-2$)',fontsize=15)
    axs[i].set_title(inputs[i],fontsize=15)
plt.tight_layout()
plt.savefig('./preds_5seeds/scatterPreds_M8.pdf')
plt.show()

## Predictions with original seed and 5 inputs

In [ ]:
seedWeights=73

#Create MLP with initial layer adapted to new number of inputs
I1=Input(shape=(trainfi_Vx.shape[1],))
I2=Input(shape=(trainfi_Vy.shape[1],))
I3=Input(shape=(trainfi_Vz.shape[1],))

h1_p1=Dense(trainfi_Vx.shape[1],activation=leakyrelu)(I1)
h1_p2=Dense(trainfi_Vy.shape[1],activation=leakyrelu)(I2)
h1_p3=Dense(trainfi_Vz.shape[1],activation=leakyrelu)(I3)
h1=Concatenate()([h1_p1,h1_p2,h1_p3])
h2=Dense(500,activation=leakyrelu)(h1)
hdp1=keras.layers.Dropout(0.5)(h2)
h3=Dense(420,activation=leakyrelu)(hdp1)
hdp2=keras.layers.Dropout(0.35)(h3)
h4=Dense(360,activation=leakyrelu)(hdp2)
hdp3=keras.layers.Dropout(0.25)(h4)
h5=Dense(300,activation=leakyrelu)(hdp3)
hdp4=keras.layers.Dropout(0.15)(h5)
h6=Dense(150,activation=leakyrelu)(hdp4)
h7=Dense(75,activation=leakyrelu)(h6)
h8=Dense(32,activation=leakyrelu)(h7)
Out=Dense(3)(h8)

model=keras.models.Model(inputs=[I1,I2,I3],outputs=Out)
model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=best_params[0]), metrics=['mae'])

set_seed(seedWeights)
reinitialize_weights(model, seedWeights)

earlystop = EarlyStopping(monitor='val_loss', patience=100, min_delta=0.0005, verbose=1, restore_best_weights = False)
callbacks_list = [earlystop]

history = model.fit([trainfi_Vx_pd,trainfi_Vy_pd,trainfi_Vz_pd], trainfl_pd,epochs=10000,
batch_size=int(best_params[1]),verbose=0,validation_split = best_params[2],callbacks=callbacks_list)

loss=model.evaluate([testfi_Vx_pd,testfi_Vy_pd,testfi_Vz_pd], testfl_pd)
predicciones=model.predict([testfi_Vx_pd,testfi_Vy_pd,testfi_Vz_pd])

In [ ]:
scaledPred=predicciones*5000
np.savetxt('./preds_originalSeed/Preds_M8.csv', scaledPred, delimiter=';', fmt='%0.6f')

## Noise sensitivity

In [ ]:
#Train model with best parameters
loses=np.ones(5)*1000
m=np.zeros(3)
R2=np.zeros(3)
seedWeights=73
seedsCases=[13,37,258,456,1379]

noiseLevels=np.array([0.01,0.025,0.05,0.1])

pendientes=np.zeros((4,5,100))
Rcuadrados=np.zeros((4,5,100))

for n in range(4):
    for i in range(5):
        #Select different cases for train and test according to seed
        train_final_input,test_final_input,train_final_label,test_final_label = random_split(datos,labels,seedsCases[i])

        train_final_input=scaler.transform(train_final_input)
        test_final_input=scaler.transform(test_final_input)

        trainfi_Vx=np.array(train_final_input[:,0:inputs-2:3])
        trainfi_Vy=np.array(train_final_input[:,1:inputs-1:3])
        trainfi_Vz=np.array(train_final_input[:,2:inputs:3])

        testfi_Vx=np.array(test_final_input[:,0:inputs-2:3])
        testfi_Vy=np.array(test_final_input[:,1:inputs-1:3])
        testfi_Vz=np.array(test_final_input[:,2:inputs:3])

        trainfi_Vx_pd=pd.DataFrame(trainfi_Vx)
        trainfi_Vy_pd=pd.DataFrame(trainfi_Vy)
        trainfi_Vz_pd=pd.DataFrame(trainfi_Vz)
        testfi_Vx_pd=pd.DataFrame(testfi_Vx)
        testfi_Vy_pd=pd.DataFrame(testfi_Vy)
        testfi_Vz_pd=pd.DataFrame(testfi_Vz)

        trainfl_pd=pd.DataFrame(train_final_label)
        testfl_pd=pd.DataFrame(test_final_label)

        #Create MLP with initial layer adapted to new number of inputs
        I1=Input(shape=(trainfi_Vx.shape[1],))
        I2=Input(shape=(trainfi_Vy.shape[1],))
        I3=Input(shape=(trainfi_Vz.shape[1],))

        h1_p1=Dense(trainfi_Vx.shape[1],activation=leakyrelu)(I1)
        h1_p2=Dense(trainfi_Vy.shape[1],activation=leakyrelu)(I2)
        h1_p3=Dense(trainfi_Vz.shape[1],activation=leakyrelu)(I3)
        h1=Concatenate()([h1_p1,h1_p2,h1_p3])
        h2=Dense(500,activation=leakyrelu)(h1)
        hdp1=keras.layers.Dropout(0.5)(h2)
        h3=Dense(420,activation=leakyrelu)(hdp1)
        hdp2=keras.layers.Dropout(0.35)(h3)
        h4=Dense(360,activation=leakyrelu)(hdp2)
        hdp3=keras.layers.Dropout(0.25)(h4)
        h5=Dense(300,activation=leakyrelu)(hdp3)
        hdp4=keras.layers.Dropout(0.15)(h5)
        h6=Dense(150,activation=leakyrelu)(hdp4)
        h7=Dense(75,activation=leakyrelu)(h6)
        h8=Dense(32,activation=leakyrelu)(h7)
        Out=Dense(3)(h8)

        model=keras.models.Model(inputs=[I1,I2,I3],outputs=Out)
        model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=best_params[0]), metrics=['mae'])

        set_seed(seedWeights)
        reinitialize_weights(model, seedWeights)

        earlystop = EarlyStopping(monitor='val_loss', patience=100, min_delta=0.0005, verbose=1, restore_best_weights = False)
        callbacks_list = [earlystop]

        history = model.fit([trainfi_Vx_pd,trainfi_Vy_pd,trainfi_Vz_pd], trainfl_pd,epochs=10000,
        batch_size=int(best_params[1]),verbose=0,validation_split = best_params[2],callbacks=callbacks_list)

        start=time()

        for t in range(100):

            train_final_input,test_final_input,train_final_label,test_final_label = random_split(datos,labels,seedsCases[i])
            noise=np.random.normal(loc=0.0, scale=noiseLevels[n],size=test_final_input.size).reshape(test_final_input.shape)
            noise += 1 
            newDatos=test_final_input * noise
            test_final_input=scaler.transform(newDatos)

            testfi_Vx=np.array(test_final_input[:,0:inputs-2:3])
            testfi_Vy=np.array(test_final_input[:,1:inputs-1:3])
            testfi_Vz=np.array(test_final_input[:,2:inputs:3])
            testfi_Vx_pd=pd.DataFrame(testfi_Vx)
            testfi_Vy_pd=pd.DataFrame(testfi_Vy)
            testfi_Vz_pd=pd.DataFrame(testfi_Vz)

            predicciones=model.predict([testfi_Vx_pd,testfi_Vy_pd,testfi_Vz_pd],verbose=False)

            for j in range(3):
                regression=LinearRegression().fit(test_final_label[:,j].reshape(-1,1),predicciones[:,j].reshape(-1,1))
                R2[j]=regression.score(test_final_label[:,j].reshape(-1,1),predicciones[:,j].reshape(-1,1))
                m[j]=regression.coef_

            pendientes[n,i,t]=np.mean(m)
            Rcuadrados[n,i,t]=np.mean(R2) 
            m=np.zeros(3)
            R2=np.zeros(3)

        print(f'Noise level evaluated in: {round(time()-start,3)} seconds')



    

In [ ]:
np.savez('./reviewInputSelection/noise/M8_1inputs_pendiente.npz',pendientes)
np.savez('./reviewInputSelection/noise/M8_1inputs_Rcuadrado.npz',Rcuadrados)